In [10]:
import os

!git clone https://github.com/sp-uhh/sgmse.git
os.chdir('/kaggle/working/sgmse')

!pip install -r requirements.txt --quiet
!pip install pesq pystoi pandas gdown --quiet

fatal: destination path 'sgmse' already exists and is not an empty directory.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 46.6 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires typeguard<5,>=4, but you have typeguard 2.13.3 which is incompatible.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.


In [ ]:
!pip install numpy==1.26.4 --force-reinstall --quiet
# restart the session after this

In [11]:
import os
os.chdir('/kaggle/working/sgmse')
import numpy as np
print(np.__version__)  # should be 1.26.4

1.26.4


In [ ]:
!gdown 1_H3EXvhcYBhOZ9QNUcD5VZHc6ktrRbwQ -O voicebank_pretrained.ckpt

In [12]:
patch = """
import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load
"""

with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
    content = f.read()

content = content.replace('import torch\n', 'import torch\n' + patch, 1)

with open('/kaggle/working/sgmse/enhancement.py', 'w') as f:
    f.write(content)

print("enhancement.py patched")

enhancement.py patched


In [13]:
# APPLYING PHYSICS LOSS

physics_loss_fn = '''
    def _wave_equation_loss(self, x_td):
        d1 = x_td[:, 1:] - x_td[:, :-1]
        d2 = d1[:, 1:] - d1[:, :-1]
        return torch.mean(d2 ** 2)

'''

old_loss_line = "                loss = losses_tf + self.l1_weight * losses_l1\n"
new_loss_line = """                # physics loss
                losses_wave = self._wave_equation_loss(x_hat_td)
                loss = losses_tf + self.l1_weight * losses_l1 + self.physics_weight * losses_wave\n"""

old_init_line = "        self.loss_type = loss_type\n"
new_init_line = "        self.loss_type = loss_type\n        self.physics_weight = 0.1\n"

with open('/kaggle/working/sgmse/sgmse/model.py', 'r') as f:
    content = f.read()

content = content.replace('    def _loss(self', physics_loss_fn + '    def _loss(self', 1)
content = content.replace(old_init_line, new_init_line, 1)
content = content.replace(old_loss_line, new_loss_line, 1)

with open('/kaggle/working/sgmse/sgmse/model.py', 'w') as f:
    f.write(content)

# verify
!grep -n "physics\|wave_equation" /kaggle/working/sgmse/sgmse/model.py

72:        self.physics_weight = 0.1
73:        self.physics_weight = 0.1
74:        self.physics_weight = 0.1
75:        self.physics_weight = 0.1
135:    def _wave_equation_loss(self, x_td):
142:    def _wave_equation_loss(self, x_td):
203:                # physics loss
204:                losses_wave = self._wave_equation_loss(x_hat_td)
205:                loss = losses_tf + self.l1_weight * losses_l1 + self.physics_weight * losses_wave


In [14]:
import soundfile as sf
import numpy as np
from datasets import load_dataset, Audio

TEST_DIR = "data/test"
TRAIN_DIR = "data/train"
os.makedirs(f"{TEST_DIR}/clean", exist_ok=True)
os.makedirs(f"{TEST_DIR}/noisy", exist_ok=True)
os.makedirs(f"{TRAIN_DIR}/clean", exist_ok=True)
os.makedirs(f"{TRAIN_DIR}/noisy", exist_ok=True)

print("Loading test set...")
test_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="test")
test_data = test_data.cast_column("clean", Audio(sampling_rate=16000))
test_data = test_data.cast_column("noisy", Audio(sampling_rate=16000))

for i, sample in enumerate(test_data):
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TEST_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TEST_DIR}/noisy/{fname}", noisy, 16000)
print(f"Wrote {len(test_data)} test samples")

print("Loading train set (500 samples)...")
train_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="train")
train_data = train_data.cast_column("clean", Audio(sampling_rate=16000))
train_data = train_data.cast_column("noisy", Audio(sampling_rate=16000))

for i in range(1250):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TRAIN_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TRAIN_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 1250 train samples")

VALID_DIR = "data/valid"
os.makedirs(f"{VALID_DIR}/clean", exist_ok=True)
os.makedirs(f"{VALID_DIR}/noisy", exist_ok=True)

print("Writing validation set (100 samples)...")
for i in range(1250, 1350):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{VALID_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{VALID_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 100 validation samples")

Loading test set...
Wrote 824 test samples
Loading train set (500 samples)...
Wrote 1250 train samples
Writing validation set (100 samples)...
Wrote 100 validation samples


In [22]:
finetune_script = '''
import os
os.chdir('/kaggle/working/sgmse')
import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
from sgmse.model import ScoreModel
from sgmse.data_module import SpecsDataModule

CKPT_PATH = "/kaggle/working/sgmse/voicebank_pretrained.ckpt"
SAVE_DIR = "/kaggle/working/sgmse_physics_finetuned"
DATA_DIR = "/kaggle/working/sgmse/data"
EPOCHS = 10
LR = 1e-5
os.makedirs(SAVE_DIR, exist_ok=True)

model = ScoreModel.load_from_checkpoint(
    CKPT_PATH,
    map_location="cuda",
    loss_type="data_prediction",
    loss_weighting="1",
    l1_weight=0.001,
    pesq_weight=0.0,
    num_eval_files=0,
    lr=LR
)
model.physics_weight = 0.001

class PrintValLoss(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        val_loss = metrics.get("valid_loss")
        train_loss = metrics.get("train_loss_epoch")
        if val_loss is not None and train_loss is not None:
            print("\\n[Epoch " + str(trainer.current_epoch) + "] train_loss=" + str(round(float(train_loss), 4)) + " | valid_loss=" + str(round(float(val_loss), 4)) + "\\n")
        elif val_loss is not None:
            print("\\n[Epoch " + str(trainer.current_epoch) + "] valid_loss=" + str(round(float(val_loss), 4)) + "\\n")

data_module = SpecsDataModule(
    base_dir=DATA_DIR,
    format="default",
    batch_size=4,
    n_fft=510,
    hop_length=128,
    num_frames=256,
    window="hann",
    num_workers=2,
    dummy=False,
    spec_factor=0.15,
    spec_abs_exponent=0.5,
    normalize="noisy",
    transform_type="exponent"
)

checkpoint_callback = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="physics_epoch{epoch:02d}_valloss{valid_loss:.4f}",
    save_top_k=3,
    monitor="valid_loss",
    mode="min",
    every_n_epochs=1
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_callback, PrintValLoss()],
    log_every_n_steps=10,
    enable_progress_bar=True
)

trainer.fit(model, datamodule=data_module)
print("Best checkpoint: " + checkpoint_callback.best_model_path)
'''

with open('/kaggle/working/sgmse/finetune_physics.py', 'w') as f:
    f.write(finetune_script)

print("Script written successfully")

Script written successfully


In [16]:
with open('/kaggle/working/sgmse/sgmse/model.py', 'r') as f:
    lines = f.readlines()

# find and remove the duplicate and replace with fixed version
new_lines = []
skip_next = 0
i = 0
while i < len(lines):
    if '_wave_equation_loss' in lines[i] and 'def ' in lines[i]:
        # skip this function definition (4 lines: def, d1, d2, return)
        if skip_next == 0:
            # first occurrence - skip it entirely
            skip_next = 1
            i += 5  # skip def + d1 + d2 + return + blank line
            continue
        else:
            # second occurrence - replace with fixed version
            new_lines.append('    def _wave_equation_loss(self, x_td):\n')
            new_lines.append('        if x_td.dim() == 1:\n')
            new_lines.append('            x_td = x_td.unsqueeze(0)\n')
            new_lines.append('        d1 = x_td[:, 1:] - x_td[:, :-1]\n')
            new_lines.append('        d2 = d1[:, 1:] - d1[:, :-1]\n')
            new_lines.append('        return torch.mean(d2 ** 2)\n')
            i += 5
            continue
    new_lines.append(lines[i])
    i += 1

with open('/kaggle/working/sgmse/sgmse/model.py', 'w') as f:
    f.writelines(new_lines)

# verify
!grep -n -A 7 "_wave_equation_loss" /kaggle/working/sgmse/sgmse/model.py

137:    def _wave_equation_loss(self, x_td):
138-        if x_td.dim() == 1:
139-            x_td = x_td.unsqueeze(0)
140-        d1 = x_td[:, 1:] - x_td[:, :-1]
141-        d2 = d1[:, 1:] - d1[:, :-1]
142-        return torch.mean(d2 ** 2)
143-    def _loss(self, forward_out, x_t, z, t, mean, x):
144-        """
--
200:                losses_wave = self._wave_equation_loss(x_hat_td)
201-                loss = losses_tf + self.l1_weight * losses_l1 + self.physics_weight * losses_wave
202-        else:
203-            raise ValueError("Invalid loss type: {}".format(self.loss_type))
204-
205-        return loss
206-
207-    def _step(self, batch, batch_idx):


In [23]:
!python finetune_physics.py

Lightning automatically upgraded your loaded checkpoint from v1.5.10 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint voicebank_pretrained.ckpt`
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-05 05:23:30.252320: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772688210.270733     528 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772688210.276790     528 cuda_blas.cc:1407] Unable to register cuBLAS facto

In [ ]:
!sed -i 's/weights_only: Optional\[bool\] = None,/weights_only: Optional[bool] = False,/' /usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py

!sed -i 's/weights_only: Optional\[bool\] = None,/weights_only: Optional[bool] = False,/' /usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py

# verify both patches
!grep -n "weights_only" /usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py
!grep -n "weights_only" /usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py

In [ ]:
with open('sanity_check.py', 'w') as f:
    f.write("""
import os
os.chdir('/kaggle/working/sgmse')

# patch torch.load before anything else imports it
import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

from sgmse.model import ScoreModel

model = ScoreModel.load_from_checkpoint(
    'voicebank_pretrained.ckpt',
    map_location='cpu',
    loss_type='data_prediction',
    loss_weighting='1',
    l1_weight=0.001,
    pesq_weight=0.0,
    num_eval_files=0,
    lr=1e-5
)
print('model loaded ok')
print('physics_weight:', model.physics_weight)
""")

!python sanity_check.py

In [28]:
with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
    content = f.read()

# Remove all the duplicate patch blocks, keep only one
import re
patch_block = """import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load"""

# Remove all occurrences
while patch_block in content:
    content = content.replace(patch_block, '')

# Add a single clean patch at the top
clean_patch = """import glob
import torch

_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

"""

# Remove any stray duplicate 'import glob' and 'import torch' at top
content = content.lstrip()
content = re.sub(r'^(import glob\n|import torch\n)+', '', content)
content = clean_patch + content

with open('/kaggle/working/sgmse/enhancement.py', 'w') as f:
    f.write(content)

print("Fixed. New top of file:")
with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
    print(f.read()[:400])

Fixed. New top of file:
import glob
import torch

_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load









from tqdm import tqdm
from os import makedirs
from soundfile import write
from torchaudio import load
from os.path import join, dirname
from argparse import ArgumentParser



In [29]:
CKPT = "/kaggle/working/sgmse_physics_finetuned/physics_epochepoch=09_vallossvalid_loss=25.3103.ckpt"

!python enhancement.py \
    --test_dir data/test/noisy \
    --enhanced_dir enhanced_physics \
    --ckpt $CKPT \
    --N 10

Set TORCH_CUDA_ARCH_LIST to: 6.0
100%|█████████████████████████████████████████| 824/824 [29:05<00:00,  2.12s/it]


In [31]:
!python calc_metrics.py \
    --clean_dir data/test/clean \
    --noisy_dir data/test/noisy \
    --enhanced_dir enhanced_physics

100%|█████████████████████████████████████████| 824/824 [03:36<00:00,  3.80it/s]
PESQ: 1.04 ± 0.01
ESTOI: 0.19 ± 0.06
SI-SDR: -27.2 ± 11.2
SI-SIR: 10.3 ± 14.7
SI-SAR: -27.2 ± 11.2


In [32]:
import torch
model_state = torch.load('/kaggle/working/sgmse_physics_finetuned/physics_epochepoch=09_vallossvalid_loss=25.3103.ckpt', weights_only=False)
layers = list(model_state['state_dict'].keys())
print(f"Total layers: {len(layers)}")
print("First 10:", layers[:10])
print("Last 10:", layers[-10:])

Total layers: 647
First 10: ['dnn.output_layer.weight', 'dnn.output_layer.bias', 'dnn.all_modules.0.W', 'dnn.all_modules.1.weight', 'dnn.all_modules.1.bias', 'dnn.all_modules.2.weight', 'dnn.all_modules.2.bias', 'dnn.all_modules.3.weight', 'dnn.all_modules.3.bias', 'dnn.all_modules.4.GroupNorm_0.weight']
Last 10: ['dnn.all_modules.74.GroupNorm_1.weight', 'dnn.all_modules.74.GroupNorm_1.bias', 'dnn.all_modules.74.Conv_1.weight', 'dnn.all_modules.74.Conv_1.bias', 'dnn.all_modules.74.Conv_2.weight', 'dnn.all_modules.74.Conv_2.bias', 'dnn.all_modules.75.weight', 'dnn.all_modules.75.bias', 'dnn.all_modules.76.weight', 'dnn.all_modules.76.bias']


In [33]:
finetune_script = '''
import os
os.chdir('/kaggle/working/sgmse')
import torch
if not hasattr(torch, '_load_patched'):
    _orig = torch.load
    def _patched(*args, **kwargs):
        kwargs['weights_only'] = False
        return _orig(*args, **kwargs)
    torch.load = _patched
    torch._load_patched = True

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
from sgmse.model import ScoreModel
from sgmse.data_module import SpecsDataModule

CKPT_PATH = "/kaggle/working/sgmse/voicebank_pretrained.ckpt"
SAVE_DIR = "/kaggle/working/sgmse_physics_v2"
DATA_DIR = "/kaggle/working/sgmse/data"
LR = 5e-6
os.makedirs(SAVE_DIR, exist_ok=True)

model = ScoreModel.load_from_checkpoint(
    CKPT_PATH,
    map_location="cuda",
    loss_type="data_prediction",
    loss_weighting="1",
    l1_weight=0.001,
    pesq_weight=0.0,
    num_eval_files=0,
    lr=LR
)
model.physics_weight = 0.00001

# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze output layer and last 10 modules (67-76)
for param in model.dnn.output_layer.parameters():
    param.requires_grad = True
for i in range(67, 77):
    for param in model.dnn.all_modules[i].parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Trainable params: " + str(trainable) + " / " + str(total))

class PrintValLoss(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        val_loss = metrics.get("valid_loss")
        train_loss = metrics.get("train_loss_epoch")
        if val_loss is not None and train_loss is not None:
            print("\\n[Epoch " + str(trainer.current_epoch) + "] train_loss=" + str(round(float(train_loss), 4)) + " | valid_loss=" + str(round(float(val_loss), 4)) + "\\n")
        elif val_loss is not None:
            print("\\n[Epoch " + str(trainer.current_epoch) + "] valid_loss=" + str(round(float(val_loss), 4)) + "\\n")

data_module = SpecsDataModule(
    base_dir=DATA_DIR,
    format="default",
    batch_size=4,
    n_fft=510,
    hop_length=128,
    num_frames=256,
    window="hann",
    num_workers=2,
    dummy=False,
    spec_factor=0.15,
    spec_abs_exponent=0.5,
    normalize="noisy",
    transform_type="exponent"
)

checkpoint_callback = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="physics_v2_epoch{epoch:02d}_valloss{valid_loss:.4f}",
    save_top_k=3,
    monitor="valid_loss",
    mode="min",
    every_n_epochs=1
)

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_callback, PrintValLoss()],
    log_every_n_steps=10,
    enable_progress_bar=True,
    gradient_clip_val=1.0
)

trainer.fit(model, datamodule=data_module)
print("Best checkpoint: " + checkpoint_callback.best_model_path)
'''

with open('/kaggle/working/sgmse/finetune_physics_v2.py', 'w') as f:
    f.write(finetune_script)
print("Script written successfully")

Script written successfully


In [34]:
!python finetune_physics_v2.py

Lightning automatically upgraded your loaded checkpoint from v1.5.10 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint voicebank_pretrained.ckpt`
Trainable params: 3097362 / 65590822
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-05 07:11:10.455366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772694670.473919    1023 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772694670.479598    1023 cuda_blas.cc:

In [35]:
CKPT = "/kaggle/working/sgmse_physics_v2/physics_v2_epochepoch=09_vallossvalid_loss=0.5441.ckpt"

!python enhancement.py \
    --test_dir data/test/noisy \
    --enhanced_dir enhanced_physics_v2 \
    --ckpt $CKPT \
    --N 10

Set TORCH_CUDA_ARCH_LIST to: 6.0
100%|█████████████████████████████████████████| 824/824 [29:42<00:00,  2.16s/it]


In [36]:
!python calc_metrics.py \
    --clean_dir data/test/clean \
    --noisy_dir data/test/noisy \
    --enhanced_dir enhanced_physics_v2

100%|█████████████████████████████████████████| 824/824 [04:02<00:00,  3.39it/s]
PESQ: 1.07 ± 0.08
ESTOI: 0.02 ± 0.02
SI-SDR: -26.3 ± 1.7
SI-SIR: 5.6 ± 4.6
SI-SAR: -26.3 ± 1.7


In [38]:
import torch

ckpt = torch.load(
    '/kaggle/working/sgmse_physics_v2/physics_v2_epochepoch=09_vallossvalid_loss=0.5441.ckpt',
    weights_only=False
)
saved_keys = list(ckpt['state_dict'].keys())
print(f"Keys in checkpoint: {len(saved_keys)}")
print("First 5:", saved_keys[:5])
print("Last 5:", saved_keys[-5:])

Keys in checkpoint: 647
First 5: ['dnn.output_layer.weight', 'dnn.output_layer.bias', 'dnn.all_modules.0.W', 'dnn.all_modules.1.weight', 'dnn.all_modules.1.bias']
Last 5: ['dnn.all_modules.74.Conv_2.bias', 'dnn.all_modules.75.weight', 'dnn.all_modules.75.bias', 'dnn.all_modules.76.weight', 'dnn.all_modules.76.bias']
